# DiffeRT-Bridge: start here
**A research walkthrough using preserved results.** Run all cells without Blender,
Sionna, DiffeRT, GPU access, or network downloads. Install `requirements-notebooks.txt`
in a separate Python 3.12+ environment first; see [the notebook guide](README.md).

The analysis notebooks distinguish software checks, agreement between solvers, and
independent physical references. They replay historical analyses; they do not
claim new simulation results or measurement validation.

1. **This notebook:** project map, evidence and provenance.
2. [Native comparisons](01_native_comparison.ipynb): 23.8 GHz controls, retained failures,
   a 60 GHz vegetation example, and an optional new generated-fixture run.
3. [Multiple diffraction](02_multiple_diffraction.ipynb): path-order effects and search controls.
4. [Full-wave validation](03_fullwave_validation.ipynb): slab convergence and reduced edge models.
5. [Blender renders](04_blender_models.ipynb): inspect the models and optionally render inside Blender with Jan-Hendrik Müller's Jupyter extension.


In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "notebooks/replay.py").is_file())
sys.path.insert(0, str(ROOT / "notebooks"))
from replay import verify_data, read_json, dataframe
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
%matplotlib inline
plt.rcParams.update({"figure.figsize": (10, 4), "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True,
                     "grid.alpha": 0.2, "figure.dpi": 110})
manifest = verify_data()
print(f"Verified {len(manifest['files'])} bundled historical data files.")
from visuals import show_image, compare_renders, verify_assets, ASSETS
verify_assets()


## See the model behind each result
The pictures below use the original synthetic geometry. Red crosses mark TX,
blue crosses RX; orange lines are retained paths. Marker sizes and surface
colors are for display. All dimensions are in metres, and no ground is added.
The model index contains 22 synthetic cases; actual plant diagnostic meshes
are outside this supplement. See notebook 04 for individual case rendering.


In [ ]:
display(compare_renders(ASSETS / "focused_leaf_branch.png", ASSETS / "multiple_cluster.png",
                        ("60 GHz leaf + square branch — detail crop", "60 GHz multi-obstacle link — overview")))


## Where the completed work lives
`main` provides the extension and this analysis suite. The research branch has eleven
investigations; its scripts include explicitly experimental profiles. The project
archive preserves historical reports, results, scenes and nineteen videos.

- [Research catalog](https://github.com/fribas001/differt-bridge/blob/research/dielectric-diffraction/research/README.md)
- [Project archive and videos](https://github.com/fribas001/differt-bridge/releases/tag/project-2026-11-11)
- [Reproduction limits](https://github.com/fribas001/differt-bridge/blob/research/dielectric-diffraction/docs/reproducibility.md)

The checked-in research index records the inspected research commit. Input-presence
checks are not a substitute for complete dependency inventories or fresh scientific replay.


In [ ]:
import json
catalog = json.loads((ROOT / "research/catalog.json").read_text())
print("Research snapshot:", catalog["source_commit"])
display(pd.DataFrame(catalog["studies"])[
    ["title", "portability", "declared_prerequisites", "replay_status"]])


## Four small evidence bundles
These summaries supplement the separate DiffeRT project archive. Full solver outputs,
scenes and runtime binaries are not needed for this analysis. The data guide lists
the original study folders and omissions. Every input is checked against its SHA-256
before use; source and redistributed hashes are recorded separately.


In [ ]:
files = pd.DataFrame(manifest["files"])
display(files.groupby("study").agg(files=("path", "count"), bytes=("bytes", "sum"),
                                  redacted_locations=("path_strings_redacted", "sum")))
receipts = [
    ("Focused vegetation", read_json("focused_60ghz/verification.json")["passed"],
     "Numerical accounting; no measurement validation"),
    ("Multiple diffraction", read_json("multiple_diffraction_60ghz/verification.json")["status"],
     "Audited paths and search controls; absolute tree loss unvalidated"),
    ("23.8 GHz controls", read_json("ris_controls_23_8ghz/verification.json")["status"],
     "Ideal controls pass; difficult geometry diagnostics remain failed"),
    ("Full-wave edge controls", read_json("edge_fullwave_60ghz/verification.json")["numerical_gates_passed"],
     "Reduced 2D model; transparent-material limit fails"),
]
display(pd.DataFrame(receipts, columns=["study", "historical_check", "scope"]))


## Conventions that matter
- Positive **excess loss** means attenuation relative to the declared empty/reference field.
  Coherent power uses `|sum(a)|²`; incoherent power uses `sum(|a|²)`. They answer different questions.
- Bridge-exported path coefficients exclude propagation phase; CIR applies it exactly once.
  Historical native DiffeRT fields and Sionna coefficients require the explicit phase
  alignment used by their comparison scripts. The supplied phase differences are already aligned.
- A failed search or unavailable coefficient stays missing. It must not become a zero-error result.
- Matching two ray tracers does not prove physical accuracy. Shared Maxwell-companion outputs
  do not establish equality of native ray-tracer outputs.
- The Blender/NumPy geometry pipeline does not preserve an end-to-end differentiation graph.

Continue with the [native comparisons](01_native_comparison.ipynb).
